# Glint Arc Flight Planning

Arc-shaped flight paths for maximizing specular solar glint on offshore targets.

To capture maximum glint, the sensor view zenith angle (VZA) must equal the solar zenith angle (SZA), and the sensor, sun, and target must lie in the same principal plane. The aircraft flies a **180 degree banked turn** at a bank angle equal to the SZA, tilting a nadir-pointing sensor to achieve the specular reflection geometry. At the arc midpoint (90 degrees into the turn), the glint angle reaches zero -- perfect specular reflection. The aircraft enters the turn on one heading and exits on the reciprocal.

The geometry is derived from **optics first, then kinematics**:
1. Specular reflection places the aircraft along the solar azimuth from the target
2. The turn center is on the **opposite side** of the aircraft from the target (toward the sun), guaranteeing the target is always **outside** the turn circle
3. The 180-degree arc is constructed around that center

The bank angle defaults to SZA when SZA <= 60 degrees. For steeper sun angles (e.g., early morning, high latitudes), an explicit `bank_angle` must be provided.

This notebook demonstrates:
1. Creating a `GlintArc` for an offshore oil/gas platform target
2. Visualizing the arc geometry and verifying the target is outside the arc
3. Computing and mapping glint angles across the sensor swath
4. Exploring how time of day affects the arc geometry
5. Comparing left vs. right bank directions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import folium
from datetime import datetime, timezone, timedelta

from hyplan.glint import GlintArc, compute_glint_arc
from hyplan.sensors import AVIRIS3
from hyplan.units import ureg
from pymap3d.vincenty import vdist

## 1. Create a Glint Arc

Define a target (offshore platform in the Gulf of Mexico) and observation parameters. The `GlintArc` computes the solar geometry, required bank angle, turn radius, and arc flight path automatically.

In [ ]:
# Offshore platform in the Gulf of Mexico
target_lat = 28.98
target_lon = -89.00

# Observation at 17:00 UTC (noon local CDT)
obs_time = datetime(2025, 6, 15, 17, 0, tzinfo=timezone.utc)

arc = GlintArc(
    target_lat=target_lat,
    target_lon=target_lon,
    observation_datetime=obs_time,
    altitude_msl=ureg.Quantity(10000, "foot"),
    speed=ureg.Quantity(145, "knot"),
    site_name="GOM Platform",
)

print(f"Solar zenith:        {arc.solar_zenith:.1f}")
print(f"Solar azimuth:       {arc.solar_azimuth:.1f}")
print(f"Bank angle:          {arc.bank_angle:.1f}  ({'auto from SZA' if arc.bank_angle == arc.solar_zenith else 'user override'})")
print(f"Turn radius:         {arc.turn_radius.to('meter'):.0f}")
print(f"Heading at midpoint: {arc.heading_at_midpoint:.1f}")
print(f"Arc length:          {arc.length.to('km'):.2f}")
print(f"Duration:            {arc.duration.to('second'):.0f}")

## 2. Visualize the Arc on an Interactive Map

Display the arc flight path along with the target platform location. The turn center is on the opposite side of the aircraft from the target, so the target is always outside the arc.

In [ ]:
m = folium.Map(location=[target_lat, target_lon], zoom_start=12)

# Arc flight path
arc_coords = [(lat, lon) for lon, lat in arc.geometry.coords]
folium.PolyLine(
    arc_coords, color="blue", weight=3,
    tooltip=f"Glint Arc: bank={arc.bank_angle:.1f}, R={arc.turn_radius.to('km'):.2f}",
).add_to(m)

# Target platform
folium.Marker(
    [target_lat, target_lon],
    popup=f"Target: {arc.site_name}",
    icon=folium.Icon(color="red", icon="industry", prefix="fa"),
).add_to(m)

# Aircraft midpoint (where glint is perfect)
folium.CircleMarker(
    [arc._aircraft_mid_lat, arc._aircraft_mid_lon],
    radius=5, color="green", fill=True,
    tooltip=f"Aircraft at arc midpoint (heading {arc.heading_at_midpoint:.0f})",
).add_to(m)

# Arc start/end waypoints
for wp, label in [(arc.waypoint1, "Start"), (arc.waypoint2, "End")]:
    folium.CircleMarker(
        [wp.latitude, wp.longitude],
        radius=4, color="orange", fill=True,
        tooltip=f"{label}: heading={wp.heading:.0f}",
    ).add_to(m)

m

### Geometry Validation

The turn center is placed on the opposite side of the aircraft from the target (toward the sun). This guarantees `dist(target, center) = ground_distance + R > R`, so the target is always outside the turn circle.

In [ ]:
from pymap3d.vincenty import vreckon as _vreckon

# Compute actual turn center (same method as GlintArc._compute_arc)
C_lat, C_lon = _vreckon(
    arc._aircraft_mid_lat, arc._aircraft_mid_lon,
    arc._turn_radius_m, arc.solar_azimuth,
)
dist_target_center = float(vdist(target_lat, target_lon, float(C_lat), float(C_lon))[0])
R = arc.turn_radius.to("meter").magnitude

print(f"Turn radius:            {R:.0f} m")
print(f"Target-to-center dist:  {dist_target_center:.0f} m")
print(f"Ratio d/R:              {dist_target_center / R:.2f}")
print(f"Target outside circle:  {dist_target_center > R}")

## 3. Compute Glint Angles Across the Sensor Swath

Use `compute_glint_arc` to sweep the AVIRIS-3 sensor FOV across the arc, computing glint angles at every (along-track, cross-track) position. The sensor is banked with the aircraft, so the effective view zenith in the Earth frame is `bank_angle + sensor_tilt`.

In [ ]:
sensor = AVIRIS3()
gdf = compute_glint_arc(arc, sensor)

print(f"Swath points: {len(gdf):,}")
print(f"Min glint angle: {gdf['glint_angle'].min():.2f}")
print(f"Max glint angle: {gdf['glint_angle'].max():.2f}")
print(f"Points with glint < 5:  {(gdf['glint_angle'] < 5).sum():,}")
print(f"Points with glint < 25: {(gdf['glint_angle'] < 25).sum():,}")

### Glint angle map (geographic)

Plot the glint angles on the map, colored by intensity. Points with low glint angles (strong specular reflection) appear in warm colors near the target.

In [ ]:
from matplotlib.colors import PowerNorm

fig, ax = plt.subplots(figsize=(10, 8))

sc = ax.scatter(
    gdf["target_longitude"], gdf["target_latitude"],
    c=gdf["glint_angle"], cmap="RdYlBu", s=1,
    norm=PowerNorm(gamma=0.4, vmin=0, vmax=90),
)
ax.plot(target_lon, target_lat, "r*", markersize=15, label="Target platform")

# Arc flight path
arc_lons = [c[0] for c in arc.geometry.coords]
arc_lats = [c[1] for c in arc.geometry.coords]
ax.plot(arc_lons, arc_lats, "k-", linewidth=2, label="Flight arc")

cb = plt.colorbar(sc, ax=ax, shrink=0.8)
cb.set_label("Glint Angle")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(
    f"Glint Angle Across AVIRIS-3 Swath\n"
    f"SZA={arc.solar_zenith:.1f}, Bank={arc.bank_angle:.1f}, "
    f"R={arc.turn_radius.to('km'):.1f}"
)
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

### Glint angle in sensor coordinates

View the glint angle as a function of along-track distance and sensor tilt angle. The specular "hot spot" (glint angle ~ 0) should appear at the arc midpoint and near tilt = 0 (where the view zenith equals the bank angle = SZA).

In [ ]:
gdf_at = compute_glint_arc(arc, sensor, output_geometry="along_track")

fig, ax = plt.subplots(figsize=(10, 5))

sc = ax.scatter(
    gdf_at["along_track_distance"] / 1000.0,
    gdf_at["tilt_angle"],
    c=gdf_at["glint_angle"], cmap="RdYlBu", s=2,
    norm=PowerNorm(gamma=0.4, vmin=0, vmax=90),
)
cb = plt.colorbar(sc, ax=ax, shrink=0.8)
cb.set_label("Glint Angle")

ax.set_xlabel("Along-Track Distance (km)")
ax.set_ylabel("Sensor Tilt Angle")
ax.set_title("Glint Angle in Sensor Coordinates")
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5, label="Tilt = 0 (VZA = bank angle)")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Time-of-Day Study

The solar zenith angle changes throughout the day, which changes the required bank angle and turn radius. Explore how the arc geometry varies from morning to afternoon (15:00-20:00 UTC, or 10 AM - 3 PM local CDT).

Hours where SZA > 60 are automatically skipped (bank angle would be too steep); pass `bank_angle` explicitly to override.

In [ ]:
hours_utc = np.arange(15, 21)  # 15:00-20:00 UTC
results = []

for h in hours_utc:
    t = datetime(2025, 6, 15, h, 0, tzinfo=timezone.utc)
    try:
        a = GlintArc(
            target_lat=target_lat, target_lon=target_lon,
            observation_datetime=t,
            altitude_msl=ureg.Quantity(10000, "foot"),
            speed=ureg.Quantity(145, "knot"),
        )
        results.append({
            "hour_utc": h,
            "local_time": f"{h - 5}:00",
            "sza": a.solar_zenith,
            "solar_az": a.solar_azimuth,
            "bank_angle": a.bank_angle,
            "turn_radius_km": a.turn_radius.to("km").magnitude,
            "arc_length_km": a.length.to("km").magnitude,
            "heading": a.heading_at_midpoint,
            "arc": a,
        })
    except Exception as e:
        print(f"  {h}:00 UTC -- skipped: {e}")

for r in results:
    print(
        f"{r['local_time']} CDT | SZA={r['sza']:.1f} | "
        f"Bank={r['bank_angle']:.1f} | R={r['turn_radius_km']:.1f} km | "
        f"Arc={r['arc_length_km']:.1f} km | Heading={r['heading']:.0f}"
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

local_times = [r["local_time"] for r in results]
szas = [r["sza"] for r in results]
radii = [r["turn_radius_km"] for r in results]

axes[0].plot(local_times, szas, "bo-")
axes[0].set_xlabel("Local Time (CDT)")
axes[0].set_ylabel("Solar Zenith / Bank Angle")
axes[0].set_title("Bank Angle vs. Time of Day")
axes[0].grid(True, alpha=0.3)

axes[1].plot(local_times, radii, "ro-")
axes[1].set_xlabel("Local Time (CDT)")
axes[1].set_ylabel("Turn Radius (km)")
axes[1].set_title("Turn Radius vs. Time of Day")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Arc geometry at different times

Overlay the arc paths for each hour on a single map to see how the arc rotates and changes radius throughout the day.

In [ ]:
m2 = folium.Map(location=[target_lat, target_lon], zoom_start=10)

folium.Marker(
    [target_lat, target_lon],
    popup="Target Platform",
    icon=folium.Icon(color="red", icon="industry", prefix="fa"),
).add_to(m2)

colors = ["darkblue", "blue", "green", "orange", "red", "darkred"]
for r, color in zip(results, colors):
    a = r["arc"]
    coords = [(lat, lon) for lon, lat in a.geometry.coords]
    folium.PolyLine(
        coords, color=color, weight=3,
        tooltip=f"{r['local_time']} CDT | SZA={r['sza']:.1f} | R={r['turn_radius_km']:.1f} km",
    ).add_to(m2)

m2

## 5. Compare Left vs. Right Bank

Both bank directions produce valid specular geometry. The aircraft heading differs by 180 -- this affects approach direction and may matter for operational planning.

In [ ]:
arc_right = GlintArc(
    target_lat=target_lat, target_lon=target_lon,
    observation_datetime=obs_time,
    altitude_msl=ureg.Quantity(10000, "foot"),
    speed=ureg.Quantity(145, "knot"),
    bank_direction="right",
)

arc_left = GlintArc(
    target_lat=target_lat, target_lon=target_lon,
    observation_datetime=obs_time,
    altitude_msl=ureg.Quantity(10000, "foot"),
    speed=ureg.Quantity(145, "knot"),
    bank_direction="left",
)

m3 = folium.Map(location=[target_lat, target_lon], zoom_start=12)

folium.Marker(
    [target_lat, target_lon],
    popup="Target", icon=folium.Icon(color="red", icon="industry", prefix="fa"),
).add_to(m3)

for a, color, label in [(arc_right, "blue", "Right bank"), (arc_left, "green", "Left bank")]:
    coords = [(lat, lon) for lon, lat in a.geometry.coords]
    folium.PolyLine(
        coords, color=color, weight=3,
        tooltip=f"{label}: heading={a.heading_at_midpoint:.0f}",
    ).add_to(m3)

print(f"Right bank heading: {arc_right.heading_at_midpoint:.0f}")
print(f"Left bank heading:  {arc_left.heading_at_midpoint:.0f}")
m3

## 6. Export

Export the arc geometry and glint swath data to GeoJSON for use in GIS software.

In [ ]:
import json

with open("glint_arc_path.geojson", "w") as f:
    json.dump(arc.to_geojson(), f, indent=2)

gdf.to_file("glint_arc_swath.geojson", driver="GeoJSON")

print("Exported: glint_arc_path.geojson, glint_arc_swath.geojson")